# Experiment 43 — Backward-free SparseWalker on ML-1M (fast v2)

Direct transfer of the successful Experiment 39 local-contrastive learner from Beauty to ML-1M. The learning rule is unchanged; this version only removes execution waste.

- corrected SparseWalker v1.1
- 65,536 concepts, K=8 active, degree=4, two graph hops
- same local contrastive / competitive / predictive updates as Experiment 39
- batch **512** (same as the successful Beauty run)
- length-bucketed batches
- padded users are removed before router/graph work at each timestep
- leave-two-out, full-catalog evaluation, seen-item masking
- **no optimizer, no backward(), no autograd gradients, no warm start**
- progress is printed during every epoch and `last.pt` is crash-safe


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, json, torch
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='research/active'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO],check=True)
for p in [f'{REPO}/src',f'{REPO}/experiments']:
    if p not in sys.path: sys.path.insert(0,p)
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('TORCH',torch.__version__)
print('BRANCH',BRANCH)
assert torch.cuda.is_available(), 'GPU runtime required'


## Run

The old notebook used batch 128 simply to mirror the gradient ML-1M baseline. That was unnecessary for local learning and multiplied Python recurrent-loop overhead. This run uses the successful Beauty batch size, 512.

You should see `ML1M_LC_PROGRESS` lines after batches 1, 2, 4, ... so the epoch never appears stuck.


In [ ]:
RESUME=False
EPOCHS=70
SCRIPT=f'{REPO}/experiments/run_ml1m_local_contrastive_walker.py'
cmd=[sys.executable,'-u',SCRIPT,
     '--epochs',str(EPOCHS),
     '--batch-size','512',
     '--eval-batch-size','1024',
     '--eval-every','1',
     '--progress-every','2']
if RESUME: cmd.append('--resume')
print('RUNNING',' '.join(cmd),flush=True)
subprocess.run(cmd,cwd=REPO,check=True)


## Inspect trajectory

The headline metric is full-catalog NDCG@10. `padding_efficiency`, `mean_batch_max_len`, and `positions_per_s` also show whether the fast path is behaving as intended.


In [ ]:
import pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_local_contrastive_ml1m/seed42')
hp=root/'history.json'
if hp.exists():
    h=pd.DataFrame(json.loads(hp.read_text()))
    cols=['epoch','mean_contrastive_margin','mean_positive_prob','mean_negative_prob','mean_router_confidence','mean_value_update','mean_context_error','val_NDCG@10','val_HR@10','val_MRR@10','positions_per_s','padding_efficiency','mean_batch_max_len']
    display(h[[c for c in cols if c in h.columns]])
    if len(h):
        best=h.loc[h['val_NDCG@10'].idxmax()]
        print('BEST',best.to_dict())
else:
    print('No history yet.')


## Final / crash recovery status

If Colab disconnects after a completed epoch, rerun setup, set `RESUME=True`, and rerun the training cell. There is no optimizer state to recover.


In [ ]:
rp=root/'result.json'
bp=root/'best.pt'
lp=root/'last.pt'
print('best.pt',bp.exists(),'last.pt',lp.exists(),'result.json',rp.exists())
if rp.exists():
    print(json.dumps(json.loads(rp.read_text()),indent=2))
elif lp.exists():
    ck=torch.load(lp,map_location='cpu')
    print('RECOVERABLE_FROM_EPOCH',ck['epoch'],'BEST_EPOCH',ck.get('best_epoch'),'BEST_VAL',ck.get('best'))
